# Physical AI Pointer Demo — Colab VLM Server

Qwen3-VL-2B-Instruct をGoogle Colab GPUで動かし、画像＋自然言語指示から `RED / BLUE / GREEN / NONE` を返します。

最初に **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択してください。

In [ ]:
!pip -q install -U git+https://github.com/huggingface/transformers accelerate gradio


In [ ]:
import torch
import gradio as gr
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)
print("Model loaded:", MODEL_ID)


In [ ]:
ALLOWED = ("RED", "BLUE", "GREEN", "NONE")

def normalize_target(text):
    upper = str(text).strip().upper()
    for token in ALLOWED:
        if upper == token:
            return token
    for token in ALLOWED:
        if token in upper:
            return token
    return "NONE"

def classify(image, instruction):
    if image is None or not str(instruction).strip():
        return "NONE"

    prompt = f"""
画像には赤・青・緑のカードが置かれています。
ユーザーの指示は「{instruction}」です。
画像と指示の両方を確認し、指すべきカードを1つ選んでください。
回答は RED / BLUE / GREEN / NONE のどれか1語だけにしてください。
対象が一意に決まらない場合は NONE としてください。
"""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
        )

    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return normalize_target(output)

print("classify() ready")


## 単体テスト

まず下のセルで画像を1枚アップロードし、Colab内だけでVLMが動くことを確認します。赤・青・緑のカードを並べた写真を使ってください。

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
filename = next(iter(uploaded))
test_image = Image.open(filename).convert("RGB")

instruction = "一番左のカードを指して"
result = classify(test_image, instruction)
print("instruction:", instruction)
print("result:", result)


## Windows PCから呼び出すためのGradioサーバー

単体テストが成功してから実行します。表示される `https://...gradio.live` URL を Windows 側 `pc/config.py` の `COLAB_GRADIO_URL` に貼り付けます。

Colabランタイムを再起動するとURLが変わる可能性があります。

In [ ]:
demo = gr.Interface(
    fn=classify,
    inputs=[
        gr.Image(type="pil", label="Camera image"),
        gr.Textbox(label="Instruction"),
    ],
    outputs=gr.Textbox(label="Target"),
    api_name="classify",
    title="Physical AI Pointer Demo — Qwen3-VL",
)

demo.launch(share=True, debug=False)
